# 10M run-independent mDST production validation

## tl;dr

- Exhaustive validation covers **10,000,000 events** in **2,000 shards** with **10,000,000 unique event UIDs**.
- The campaign uses schema `direct-mdst-tree-v4`, source commit `f4e54df23b5c`, and KLM scope `included`.
- Completion markers valid: `True`; missing shards: `0`.
- The remaining cells independently reconstruct shard/resource distributions, inspect log and metadata coverage, and render representative event trees.

## Context & Methods

This is a technical companion to the production report. The authoritative campaign validator reads every shard and verifies hashes, provenance, exact source ranges, event counts, and global UID uniqueness. This notebook then aggregates all per-shard result/metadata sidecars in bounded memory and reads only selected events for topology pictures.

### Key Assumptions

- A shard is usable only when Parquet, `.metadata.json`, `.complete`, and `.result.json` agree with its immutable manifest task.
- Non-whitespace stderr is treated as a review item.
- Event-tree pictures are representative examples; numeric campaign checks are exhaustive.

In [1]:
from __future__ import annotations
from collections import Counter, defaultdict
from itertools import islice
import json, os, re, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PRODUCTION_ROOT = Path('/data/dust/user/boyangyu/hypertagging/production_10m_ri_all_exp_20260811_f4e54df')
REPO_ROOT = Path('/data/dust/user/boyangyu/hypertagging/production_10m_ri_all_exp_20260811_f4e54df/source/f4e54df23b5c')
MANIFEST = PRODUCTION_ROOT / 'manifests' / 'mdst_10m_ri_all_exp.jsonl'
FINAL_VALIDATION = PRODUCTION_ROOT / 'validation' / 'final_validation.json'
FIGURE_DIR = PRODUCTION_ROOT / 'validation' / 'notebook' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT / 'src'))
from hypertagging.preprocessing.schema_v4 import iter_event_records_v4
records = [json.loads(line) for line in MANIFEST.read_text().splitlines() if line.strip()]
validation = json.loads(FINAL_VALIDATION.read_text())
print({'manifest_tasks': len(records), 'validated_events': validation['validated_events'], 'campaign_id': validation['campaign_id'], 'figures': str(FIGURE_DIR)})

{'manifest_tasks': 2000, 'validated_events': 10000000, 'campaign_id': 'campaign-fb070c6c9805-f4e54df23b5c', 'figures': '/data/dust/user/boyangyu/hypertagging/production_10m_ri_all_exp_20260811_f4e54df/validation/notebook/figures'}


## Data

### 1. Reconstruct all shard-level statistics

In [2]:
rows = []
node_hist = Counter(); depth_hist = Counter(); leaf_modes = Counter(); roots = Counter(); klm_by_category = Counter()
missing_companions = []
max_node_event = None; max_depth_event = None
for record in records:
    parquet = Path(record['output_file'])
    companion_paths = {
        'parquet': parquet, 'metadata': Path(str(parquet) + '.metadata.json'),
        'marker': Path(str(parquet) + '.complete'), 'result': Path(str(parquet) + '.result.json')}
    absent = [name for name, path in companion_paths.items() if not path.is_file()]
    if absent: missing_companions.append({'task_id': record['task_id'], 'missing': absent})
    result = json.loads(companion_paths['result'].read_text())
    counts = result.get('node_counts', [])
    depths = result.get('max_depths', [])
    node_hist.update(counts); depth_hist.update(depths)
    leaf_modes.update(result.get('actual_leaf_mode_distribution', {}))
    roots.update(result.get('b_root_distribution', {}))
    klm_by_category[record['physics_category']] += int(result.get('klm_nodes', 0))
    if counts:
        local = max(enumerate(counts), key=lambda item: item[1])
        candidate = (local[1], record['task_id'], local[0], parquet)
        max_node_event = max(max_node_event, candidate) if max_node_event else candidate
    if depths:
        local = max(enumerate(depths), key=lambda item: item[1])
        candidate = (local[1], record['task_id'], local[0], parquet)
        max_depth_event = max(max_depth_event, candidate) if max_depth_event else candidate
    rows.append({
        'task_id': record['task_id'], 'category': record['physics_category'],
        'experiment': next(part for part in Path(record['input_file']).parts if re.fullmatch(r'e\d+', part)),
        'events': result['events'], 'output_mib': result['output_bytes'] / 2**20,
        'events_per_second': result['events_per_second'],
        'elapsed_seconds': result['elapsed_seconds'],
        'validation_seconds': result['validation_seconds'],
        'peak_rss_mib': result['peak_resident_memory_kib'] / 1024,
        'klm_nodes': result.get('klm_nodes', 0),
        'unique_event_uids': result['unique_event_uids']})
shards = pd.DataFrame(rows).sort_values('task_id').reset_index(drop=True)
display(shards.groupby('category').agg(shards=('task_id','count'), events=('events','sum'), throughput_median=('events_per_second','median'), peak_rss_p99=('peak_rss_mib', lambda x: x.quantile(.99)), output_gib=('output_mib', lambda x: x.sum()/1024)).round(3))
assert not missing_companions
assert shards.events.sum() == validation['validated_events'] == 10_000_000
assert shards.unique_event_uids.sum() == 10_000_000
assert sum(node_hist.values()) == 10_000_000
shards.to_csv(FIGURE_DIR / 'shard_metrics.csv', index=False)
print({'missing_companions': len(missing_companions), 'node_hist_events': sum(node_hist.values()), 'max_node_event': max_node_event[:3], 'max_depth_event': max_depth_event[:3]})

,shards,events,throughput_median,peak_rss_p99,output_gib
category,,,,,
ccbar,432,2160000,37.693,751.897,24.251
charged,263,1315000,26.746,802.345,19.672
ddbar,159,795000,43.691,576.694,6.577
mixed,255,1275000,27.548,817.083,19.184
ssbar,154,770000,41.818,593.838,6.697
taupair,314,1570000,62.697,500.474,7.660
uubar,423,2115000,44.858,592.607,17.558


{'missing_companions': 0, 'node_hist_events': 10000000, 'max_node_event': (119, 309, 2970), 'max_depth_event': (6, 1998, 16)}


### 2. Confirm RI-only input and experiment coverage

In [3]:
input_paths = {Path(record['input_file']) for record in records}
experiments = sorted({next(part for part in path.parts if re.fullmatch(r'e\d+', part)) for path in input_paths})
ri_violations = [str(path) for path in input_paths if 'MC16ri_run2' not in path.parts]
coverage = {
    'manifest_tasks': len(records), 'unique_input_files': len(input_paths),
    'experiments': experiments, 'categories': sorted(shards.category.unique()),
    'ri_path_violations': len(ri_violations),
    'source_commits': sorted({r['source_git_commit'] for r in records}),
    'source_states': sorted({r['source_state'] for r in records}),
    'klm_scopes': sorted({r['klm_training_scope'] for r in records})}
display(pd.Series(coverage, name='value').to_frame())
assert experiments == ['e1004'] and not ri_violations
assert coverage['source_states'] == ['clean'] and coverage['klm_scopes'] == ['included']

,value
manifest_tasks,2000
unique_input_files,1526
experiments,[e1004]
categories,"[ccbar, charged, ddbar, mixed, ssbar, taupair,..."
ri_path_violations,0
source_commits,[f4e54df23b5c60115e475c5d68df4651899d678e]
source_states,[clean]
klm_scopes,[included]


## Results

### 3. Category coverage is complete and reflects available input volume

In [4]:
category = shards.groupby('category', as_index=False).events.sum().sort_values('events')
fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(category.category, category.events / 1e6, color='#3973ac', edgecolor='#1f2937')
ax.bar_label(bars, fmt='%.3fM', padding=4, fontfamily='monospace')
ax.set(title='Validated events by physics category', xlabel='Events (millions)', ylabel='')
ax.grid(axis='x', alpha=.2); fig.tight_layout(); fig.savefig(FIGURE_DIR/'category_coverage.png', dpi=180); plt.show()

### 4. Throughput and memory remain well below the Condor requests

In [5]:
categories = sorted(shards.category.unique()); colors = plt.cm.tab10(np.linspace(0, .75, len(categories)))
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].boxplot([shards.loc[shards.category==c,'events_per_second'] for c in categories], tick_labels=categories, showfliers=False)
axes[0].set(title='Worker throughput by category', ylabel='Events / second'); axes[0].tick_params(axis='x', rotation=35); axes[0].grid(axis='y', alpha=.2)
for color, c in zip(colors, categories):
    part=shards[shards.category==c]; axes[1].scatter(part.events_per_second, part.peak_rss_mib, s=14, alpha=.55, label=c, color=color)
axes[1].axhline(8192, color='#222', ls='--', lw=1, label='8 GiB request')
axes[1].set(title='Memory versus throughput', xlabel='Events / second', ylabel='Peak RSS (MiB)'); axes[1].grid(alpha=.2); axes[1].legend(ncol=2, fontsize=8)
fig.tight_layout(); fig.savefig(FIGURE_DIR/'throughput_memory.png', dpi=180); plt.show()

### 5. Full-event topology and leaf-mode distributions

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
x=np.array(sorted(node_hist)); y=np.array([node_hist[v] for v in x]); axes[0].plot(x,y,color='#3973ac',lw=2); axes[0].fill_between(x,y,color='#3973ac',alpha=.18)
axes[0].set(title='Nodes per event', xlabel='Retained nodes', ylabel='Events'); axes[0].grid(alpha=.2)
x=np.array(sorted(depth_hist)); y=np.array([depth_hist[v] for v in x]); axes[1].bar(x,y,color='#c7832b',edgecolor='#1f2937')
axes[1].set(title='Maximum retained level', xlabel='Level', ylabel='Events'); axes[1].grid(axis='y',alpha=.2)
labels=list(leaf_modes); values=np.array([leaf_modes[k] for k in labels]); order=np.argsort(values)
axes[2].barh(np.array(labels)[order],values[order]/1e6,color='#7b6ba8',edgecolor='#1f2937')
axes[2].set(title='Leaf kinematics modes', xlabel='Nodes (millions)', ylabel=''); axes[2].grid(axis='x',alpha=.2)
fig.tight_layout(); fig.savefig(FIGURE_DIR/'topology_leaf_modes.png',dpi=180); plt.show()

### 6. KLM nodes are present across every category

In [7]:
klm=pd.Series(klm_by_category).sort_values(); fig,ax=plt.subplots(figsize=(10,5.5)); bars=ax.barh(klm.index,klm.values,color='#b45f55',edgecolor='#1f2937')
ax.bar_label(bars,fmt='%d',padding=4,fontfamily='monospace'); ax.set(title='Retained KLM nodes by category',xlabel='Nodes',ylabel=''); ax.grid(axis='x',alpha=.2)
fig.tight_layout(); fig.savefig(FIGURE_DIR/'klm_by_category.png',dpi=180); plt.show(); assert (klm>0).all()

### 7. Condor stderr and publication metadata are clean

In [8]:
log_dir=PRODUCTION_ROOT/'logs'/'condor'; err_files=sorted(log_dir.glob('*.err'))
condor=json.loads((PRODUCTION_ROOT/'validation'/'condor_monitor_summary.json').read_text())
successful_ids={str(condor['clusters']['bulk']['cluster_id']), str(condor['clusters']['successful_preflight']['cluster_id'])}
rejected_id=str(condor['clusters']['initial_preflight_rejected']['cluster_id'])
nonempty=[]
for path in err_files:
    text=path.read_text(errors='replace').strip()
    if text: nonempty.append({'path':str(path),'bytes':path.stat().st_size,'preview':text[:300]})
successful_nonempty=[item for item in nonempty if any(f'-{cluster_id}.' in Path(item['path']).name for cluster_id in successful_ids)]
rejected_nonempty=[item for item in nonempty if f'-{rejected_id}.' in Path(item['path']).name]
log_summary={'stderr_files':len(err_files),'all_non_whitespace_stderr':len(nonempty), 'successful_non_whitespace_stderr':len(successful_nonempty), 'rejected_preflight_stderr':len(rejected_nonempty), 'completion_markers':sum(Path(str(Path(r['output_file']))+'.complete').is_file() for r in records), 'metadata_sidecars':sum(Path(str(Path(r['output_file']))+'.metadata.json').is_file() for r in records), 'result_sidecars':sum(Path(str(Path(r['output_file']))+'.result.json').is_file() for r in records)}
display(pd.Series(log_summary,name='count').to_frame()); display(pd.DataFrame(nonempty).head(20))
assert log_summary['completion_markers']==len(records)==2000
assert log_summary['metadata_sidecars']==len(records) and log_summary['result_sidecars']==len(records)
assert not successful_nonempty
assert len(rejected_nonempty)==1 and condor['initial_preflight_disposition']['accepted'] is False

,count
stderr_files,2001
all_non_whitespace_stderr,1
successful_non_whitespace_stderr,0
rejected_preflight_stderr,1
completion_markers,2000
metadata_sidecars,2000
result_sidecars,2000


,path,bytes,preview
0,/data/dust/user/boyangyu/hypertagging/producti...,1567,"Traceback (most recent call last):\n File ""/a..."


### 8. Representative event trees preserve mother-above-daughter geometry

In [9]:
def event_at(path, index):
    return next(islice(iter_event_records_v4(path), index, index+1))

def draw_tree(event, ax, title):
    nodes={int(n['node_id']):n for n in event['nodes']}
    levels=defaultdict(list)
    for node in nodes.values(): levels[int(node['level'])].append(int(node['node_id']))
    pos={}
    for level, ids in sorted(levels.items()):
        ids=sorted(ids); span=max(len(ids)-1,1)
        for j,node_id in enumerate(ids): pos[node_id]=((j-(len(ids)-1)/2)/span,level)
    for node in nodes.values():
        parent=int(node['node_id'])
        for child in node.get('daughter_ids',[]):
            child=int(child)
            if child in pos: ax.plot([pos[parent][0],pos[child][0]],[pos[parent][1],pos[child][1]],color='#9ca3af',lw=.6,zorder=1)
    kind_colors={'track':'#3973ac','ecl_cluster':'#c7832b','klm_cluster':'#b45f55','composite':'#7b6ba8'}
    for node_id,node in nodes.items():
        x,y=pos[node_id]; kind=node['node_kind']; ax.scatter(x,y,s=28,color=kind_colors.get(kind,'#6b7280'),edgecolor='#1f2937',lw=.4,zorder=2)
        if len(nodes)<=75: ax.text(x,y+.07,f"{node_id}:{kind[:2]}\n{node.get('truth_pdg',node.get('pdg',''))}",ha='center',va='bottom',fontsize=4.2)
    ax.set(title=title, xlabel=f"{event['event_uid']} | nodes={len(nodes)}", ylabel='Retained level'); ax.set_xticks([]); ax.grid(axis='y',alpha=.15)
    return nodes

category_records={}
for record in records: category_records.setdefault(record['physics_category'],record)
fig,axes=plt.subplots(4,2,figsize=(16,20)); axes=axes.ravel()
for ax,(category,record) in zip(axes,sorted(category_records.items())):
    event=event_at(Path(record['output_file']),0); draw_tree(event,ax,category)
axes[-1].axis('off'); fig.suptitle('Representative event topology from every physics category',fontsize=16,y=.995); fig.tight_layout(); fig.savefig(FIGURE_DIR/'representative_category_trees.png',dpi=180,bbox_inches='tight'); plt.show()
special=[('maximum nodes',max_node_event),('maximum depth',max_depth_event)]
fig,axes=plt.subplots(2,1,figsize=(18,15))
for ax,(label,(_,task_id,index,path)) in zip(axes,special): draw_tree(event_at(path,index),ax,f'{label}: task {task_id}, event offset {index}')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'extreme_topology_trees.png',dpi=200,bbox_inches='tight'); plt.show()

## Takeaways

In [10]:
takeaways = {
 'validated_events': int(validation['validated_events']),
 'unique_event_uids': int(validation['unique_event_uids']),
 'shards': int(validation['completed_shards']),
 'categories': validation['category_distribution'],
 'experiments': experiments, 'unique_input_files': len(input_paths),
 'output_gib': float(validation['output_bytes'])/2**30,
 'bytes_per_event': float(validation['output_bytes_per_event']),
 'throughput_median_events_per_second': float(shards.events_per_second.median()),
 'peak_rss_p99_mib': float(shards.peak_rss_mib.quantile(.99)),
 'successful_non_whitespace_stderr': len(successful_nonempty), 'rejected_preflight_stderr': len(rejected_nonempty), 'missing_companions': len(missing_companions),
 'klm_nodes': int(validation['klm_node_distribution']['klm_nodes']),
 'node_count_quantiles': validation['node_count_quantiles'],
 'maximum_depth_quantiles': validation['maximum_depth_quantiles']}
(FIGURE_DIR/'notebook_takeaways.json').write_text(json.dumps(takeaways,indent=2,sort_keys=True)+'\n')
display(pd.Series(takeaways,name='value').to_frame())

,value
validated_events,10000000
unique_event_uids,10000000
shards,2000
categories,"{'ccbar': 2160000, 'charged': 1315000, 'ddbar'..."
experiments,[e1004]
unique_input_files,1526
output_gib,101.598967
bytes_per_event,10909.106045
throughput_median_events_per_second,42.035318
peak_rss_p99_mib,769.729258
